1.


In [15]:
import pandas as pd
import sqlite3

# Conectamos con la base de datos
connection = sqlite3.connect("cervezas_data.db")

# Creamos un cursor para ejecutar consultas
crsr = connection.cursor()



In [16]:
conexion = sqlite3.connect("cervezas.db")
conexion.execute("PRAGMA foreign_keys = ON")
cursor = conexion.cursor()

cursor.executescript("""
CREATE TABLE IF NOT EXISTS CERVEZAS (
CodC TEXT PRIMARY KEY,
Envase TEXT,
Capacidad REAL,
Stock INTEGER
);

CREATE TABLE IF NOT EXISTS BARES (
CodB TEXT PRIMARY KEY,
Cif TEXT,
Nombre TEXT,
Localidad TEXT
);

CREATE TABLE IF NOT EXISTS EMPLEADOS (
CodE INTEGER PRIMARY KEY,
Nombre TEXT,
Sueldo REAL
);

CREATE TABLE IF NOT EXISTS REPARTO (
CodE INTEGER,
CodB TEXT,
CodC TEXT,
Fecha TEXT,
Cantidad INTEGER,
FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
FOREIGN KEY (CodB) REFERENCES BARES(CodB),
FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
);
""")


conexion.commit()

In [20]:
cursor.executescript("""
INSERT INTO CERVEZAS (CodC, Envase, Capacidad, Stock) VALUES
('01', 'Botella', 0.2, 3600),
('02', 'Botella', 0.33, 1200),
('03', 'Lata', 0.33, 2400),
('04', 'Botella', 1, 288),
('05', 'Barril', 60, 30);

INSERT INTO BARES (CodB, Cif, Nombre, Localidad) VALUES
('001', '11111111X', 'Stop', 'Villa Botijo'),
('002', '22222222Y', 'Las Vegas', 'Villa Botijo'),
('003', '-', 'Club Social', 'Las Ranas'),
('004', '33333333Z', 'Otra Ronda', 'La Esponja');

INSERT INTO EMPLEADOS (CodE, Nombre, Sueldo) VALUES
(1, 'Prudencio Caminero', 120000),
(2, 'Vicente Merario', 110000),
(3, 'Valentin Siempre', 100000);

INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad) VALUES
(1, '001', '01', '2005-10-21', 240),
(1, '001', '02', '2005-10-21', 48),
(1, '002', '03', '2005-10-22', 60),
(1, '004', '05', '2005-10-22', 4),
(2, '002', '03', '2005-10-22', 48),
(2, '002', '05', '2005-10-23', 2),
(2, '004', '01', '2005-10-23', 480),
(2, '004', '02', '2005-10-24', 72),
(3, '003', '03', '2005-10-24', 48),
(3, '003', '04', '2005-10-25', 20);
""")

conexion.commit()

In [18]:
cursor.executescript("""
DELETE FROM REPARTO;
DELETE FROM CERVEZAS;
DELETE FROM BARES;
DELETE FROM EMPLEADOS;
""")

In [22]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

1. Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005.

In [26]:
query_1 = '''SELECT EMPLEADOS.Nombre
FROM EMPLEADOS
JOIN REPARTO
ON REPARTO.CodE = EMPLEADOS.CodE
JOIN BARES
ON REPARTO.CodB = BARES.CodB
WHERE BARES.Nombre = 'Stop'
AND REPARTO.Fecha BETWEEN '2005-10-17' AND '2005-10-23' '''

sql_query(query_1)

,Nombre
0,Prudencio Caminero
1,Prudencio Caminero


2. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo **Botella** y capacidad inferior a 1 litro, ordenados por localidad.

In [36]:
query_2 = '''SELECT DISTINCT BARES.Nombre, BARES.Cif
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
WHERE CERVEZAS.Envase = 'Botella'
AND CERVEZAS.Capacidad < 1
ORDER BY BARES.Localidad '''

sql_query(query_2)

,Nombre,Cif
0,Otra Ronda,33333333Z
1,Stop,11111111X


3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por **Prudencio Caminero**.

In [31]:
query_3 = '''SELECT BARES.Nombre, CERVEZAS.Envase, CERVEZAS.Capacidad, REPARTO.Fecha, REPARTO.Cantidad
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
JOIN EMPLEADOS
ON REPARTO.CodE = EMPLEADOS.CodE
WHERE EMPLEADOS.Nombre = 'Prudencio Caminero'
'''

sql_query(query_3)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4


4. Obtener los bares a los que se les ha repartido envases de tipo **botella** y capacidad 0.2 ó 0.33.

In [32]:
query_4 = '''SELECT BARES.Nombre, CERVEZAS.Envase, CERVEZAS.Capacidad
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
WHERE CERVEZAS.Envase = 'Botella'
AND (CERVEZAS.Capacidad = 0.2 OR  CERVEZAS.Capacidad = 0.33)
'''

sql_query(query_4)

,Nombre,Envase,Capacidad
0,Stop,Botella,0.20
1,Stop,Botella,0.33
2,Otra Ronda,Botella,0.20
3,Otra Ronda,Botella,0.33


5. Nombre de los empleados que han repartido a los bares **"Stop"** y **"Las Vegas"** cervezas con envase botella.

In [34]:
query_5 = '''SELECT BARES.Nombre, EMPLEADOS.Nombre, CERVEZAS.Envase
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
JOIN EMPLEADOS
ON REPARTO.CodE = EMPLEADOS.CodE
WHERE CERVEZAS.Envase = 'Botella'
AND (BARES.Nombre= 'Stop' OR  BARES.Nombre = 'Las Vegas')
'''

sql_query(query_5)

,Nombre,Nombre,Envase
0,Stop,Prudencio Caminero,Botella
1,Stop,Prudencio Caminero,Botella


6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de **Villa Botijo**.

In [35]:
query_6 = '''SELECT EMPLEADOS.Nombre, COUNT(REPARTO.Fecha), BARES.Localidad
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN EMPLEADOS
ON REPARTO.CodE = EMPLEADOS.CodE
WHERE BARES.Localidad != 'Villa Botijo'
GROUP BY REPARTO.Fecha
'''

sql_query(query_6)

,Nombre,COUNT(REPARTO.Fecha),Localidad
0,Prudencio Caminero,1,La Esponja
1,Vicente Merario,1,La Esponja
2,Vicente Merario,2,La Esponja
3,Valentin Siempre,1,Las Ranas


7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

In [39]:
query_7 = '''SELECT BARES.Nombre, BARES.Localidad, CERVEZAS.Capacidad * REPARTO.Cantidad
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
GROUP BY BARES.Nombre
ORDER BY (CERVEZAS.Capacidad * REPARTO.Cantidad) DESC
LIMIT 1
'''

sql_query(query_7)

,Nombre,Localidad,CERVEZAS.Capacidad * REPARTO.Cantidad
0,Otra Ronda,La Esponja,240.0


8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.

In [40]:
query_8 = '''SELECT BARES.Nombre, CERVEZAS.Envase, CERVEZAS.Capacidad
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
WHERE CERVEZAS.Envase = 'Botella'
AND CERVEZAS.Capacidad < 1
GROUP BY BARES.Nombre

'''

sql_query(query_8)

,Nombre,Envase,Capacidad
0,Otra Ronda,Botella,0.2
1,Stop,Botella,0.2


9. Subir un 5% el sueldo del empleado que más días haya trabajado.

In [41]:
query_9 = '''SELECT e.CodE, e.Nombre, COUNT(DISTINCT r.Fecha) AS DiasTrabajados
FROM EMPLEADOS AS e
JOIN REPARTO AS r ON e.CodE = r.CodE
GROUP BY e.CodE, e.Nombre
ORDER BY DiasTrabajados DESC
LIMIT 1;'''

sql_query(query_9)

,CodE,Nombre,DiasTrabajados
0,2,Vicente Merario,3


In [43]:
query_9_2 = '''UPDATE EMPLEADOS
SET Sueldo = Sueldo * 1.05
WHERE CodE = (
    SELECT CodE
    FROM REPARTO
    GROUP BY CodE
    ORDER BY COUNT(DISTINCT Fecha) DESC
    LIMIT 1
);'''



10. Insertar un nuevo reparto del empleado **"Vicente Merario"** al bar **"Stop"** de 48 cervezas de tipo lata el día 10/26/05.

In [46]:
cursor.execute('''
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES (
(SELECT CodE FROM EMPLEADOS
WHERE Nombre = 'Vicente Merario'),
(SELECT CodB FROM BARES
WHERE Nombre = 'Stop'),
(SELECT CodC FROM CERVEZAS
WHERE Envase = 'Lata'),
'2005-10-26',
48
);
''')
conexion.commit()